# Atelier 01 — Nettoyage des données automobiles

Source : `fic_etiq_edition_40-mars-2015.csv` (séparateur `;`, encodage Latin-1).
Ce notebook autonome remplace les exemples de bâtiments par un nettoyage adapté aux véhicules.
Exécuter les cellules dans l'ordre depuis le dossier de l'atelier. Seule la bibliothèque pandas est nécessaire.

Deux exports sont produits : un jeu complet nettoyé, conservant les absences explicites, et un sous-ensemble sans valeurs manquantes sur les variables choisies pour l'analyse consommation–CO₂. Les fichiers sources sont conservés.


In [1]:
from pathlib import Path
import pandas as pd

source = Path('fic_etiq_edition_40-mars-2015.csv')
sortie = Path('donnees_nettoyees')
sortie.mkdir(exist_ok=True)
df_brut = pd.read_csv(source, sep=';', encoding='latin-1')
df = df_brut.copy()
print(f"Source : {len(df):,} lignes et {len(df.columns)} colonnes")


Source : 20,880 lignes et 26 colonnes


## 1. Normaliser les textes et les dates

Retirer les espaces en début et fin de texte et convertir les chaînes vides en valeurs manquantes. Garder les identifiants comme textes, sans fusionner des variantes de modèles ou des codes d'énergie différents. Les dates ne précisent que le mois : les convertir en `AAAA-MM`, sans inventer de jour.


In [2]:
df.columns = df.columns.str.strip().str.lower()
espaces_corriges = {}
vides_detectes = {}
for col in df.select_dtypes(include=['object', 'string']).columns:
    texte = df[col].astype('string')
    propre = texte.str.strip()
    espaces_corriges[col] = int(texte.ne(propre).sum())
    vides_detectes[col] = int(propre.eq('').sum())
    df[col] = propre.replace('', pd.NA)

dates = {'juin-13': '2013-06', 'juin-14': '2014-06',
         'déc-14': '2014-12', 'mars-15': '2015-03'}
inconnues = set(df['date_maj'].dropna()) - set(dates)
if inconnues:
    raise ValueError(f'Dates à vérifier : {inconnues}')
df['date_maj'] = df['date_maj'].map(dates).astype('string')
print('Cellules avec espaces corrigés :', sum(espaces_corriges.values()))
print('Champs vides révélés :', {k: v for k, v in vides_detectes.items() if v})


Cellules avec espaces corrigés : 229680
Champs vides révélés : {'champ_v9': 56}


## 2. Doublons et cohérence physique

Supprimer uniquement les lignes entièrement identiques après normalisation. Un identifiant répété ne suffit pas à conclure à un doublon. Isoler les mesures négatives, les masses ou puissances nulles et les masses minimale/maximale inversées dans un fichier à vérifier. Conserver les zéros de consommation et d'émissions : ils ne sont pas automatiquement des erreurs.


In [3]:
doublons = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
numeriques = df.select_dtypes(include='number').columns
negatifs = df[numeriques].lt(0).any(axis=1)
non_positifs = df[['puiss_admin', 'puiss_max', 'puiss_heure',
                   'masse_ordma_min', 'masse_ordma_max']].le(0).any(axis=1)
masses_inversees = df['masse_ordma_min'].gt(df['masse_ordma_max'])
anomalies = negatifs | non_positifs | masses_inversees
a_verifier = df.loc[anomalies].copy()
a_verifier['motif'] = [
    '; '.join(motif for masque, motif in [
        (negatifs, 'mesure négative'), (non_positifs, 'masse ou puissance non positive'),
        (masses_inversees, 'masse minimale supérieure à la maximale')
    ] if masque.loc[i]) for i in a_verifier.index
]
df = df.loc[~anomalies].copy()
print('Doublons exacts supprimés :', doublons)
print('Lignes isolées pour vérification :', len(a_verifier))


Doublons exacts supprimés : 0
Lignes isolées pour vérification : 0


## 3. Traiter les absences selon leur contexte

Un dataset propre peut contenir des valeurs manquantes documentées. Ici, les absences de consommation mixte et de CO₂ concernent le code `EL` : une médiane globale attribuerait arbitrairement les caractéristiques d'autres motorisations à ces véhicules. Aucune valeur n'est donc imputée.

`puiss_heure`, `hc` et `hcnox` restent dans le jeu complet pour préserver l'information disponible. Pour l'analyse consommation–CO₂, on sélectionne explicitement les variables utiles et conserve uniquement les lignes complètes sur ces variables. Ce sous-ensemble ne représente donc pas toutes les motorisations du fichier initial.

Les valeurs extrêmes ne sont pas supprimées automatiquement : une forte puissance ou consommation peut être réelle. Pour un futur modèle ML, séparer entraînement et test avant d'apprendre une éventuelle imputation.


In [4]:
bilan = pd.DataFrame({
    'type': df.dtypes.astype(str),
    'absences_avant': df_brut.isna().sum(),
    'espaces_corriges': pd.Series(espaces_corriges),
    'vides_detectes': pd.Series(vides_detectes),
    'absences_apres': df.isna().sum(),
    'pourcentage_absent': df.isna().mean().mul(100).round(2),
})
bilan[['espaces_corriges', 'vides_detectes']] = bilan[['espaces_corriges', 'vides_detectes']].fillna(0).astype(int)
print(bilan.to_string())
print('\nAbsences de consommation par énergie :')
print(df.groupby('energ')['conso_mixte'].apply(lambda s: int(s.isna().sum())).to_string())


                      type  absences_avant  espaces_corriges  vides_detectes  absences_apres  pourcentage_absent
champ_v9            string               0             20880              56              56                0.27
cnit                string               0             20880               0               0                0.00
co2_mixte          float64              56                 0               0              56                0.27
co_typ_1           float64             275                 0               0             275                1.32
conso_exurb        float64             121                 0               0             121                0.58
conso_mixte        float64              56                 0               0              56                0.27
conso_urb_93       float64             121                 0               0             121                0.58
date_maj            string               0                 0               0               0    

## 4. Trier et construire le sous-ensemble d'analyse

Trier par marque, modèle, énergie, puissance et identifiant pour faciliter la lecture. Le jeu complet conserve les 26 variables. Le sous-ensemble contient les identifiants, les catégories et les mesures utiles à l'analyse de la consommation et du CO₂.


In [5]:
cles_tri = ['lib_mrq_doss', 'lib_mod_doss', 'energ', 'puiss_max', 'cnit']
df = df.sort_values(cles_tri, kind='stable', na_position='last').reset_index(drop=True)
colonnes_analyse = [
    'cnit', 'lib_mrq_doss', 'lib_mod_doss', 'energ', 'hybride',
    'typ_boite_nb_rapp', 'puiss_admin', 'puiss_max',
    'masse_ordma_min', 'masse_ordma_max', 'conso_mixte', 'co2_mixte'
]
incompletes = df[colonnes_analyse].isna().any(axis=1)
df_analyse = df.loc[~incompletes, colonnes_analyse].reset_index(drop=True)
print(f'Jeu complet nettoyé : {df.shape}')
print(f"Sous-ensemble complet pour l'analyse : {df_analyse.shape}")
print(f'Lignes exclues du sous-ensemble : {int(incompletes.sum())}')
print('Énergies exclues :', df.loc[incompletes, 'energ'].value_counts().to_dict())
print(df_analyse.head().to_string(index=False))


Jeu complet nettoyé : (20880, 26)
Sous-ensemble complet pour l'analyse : (20824, 12)
Lignes exclues du sous-ensemble : 56
Énergies exclues : {'EL': 56}
           cnit lib_mrq_doss lib_mod_doss energ hybride typ_boite_nb_rapp  puiss_admin  puiss_max  masse_ordma_min  masse_ordma_max  conso_mixte  co2_mixte
M10ALFVP000G340   ALFA ROMEO          159    ES     non               M 6           12      147.0             1505             1505          7.8      182.0
M10ALFVP000H341   ALFA ROMEO          159    ES     non               M 6           12      147.0             1555             1555          8.0      186.0
M10ALFVP000E302   ALFA ROMEO          159    GO     non               M 6            7      100.0             1565             1565          5.1      134.0
M10ALFVP000F303   ALFA ROMEO          159    GO     non               M 6            7      100.0             1565             1565          5.1      134.0
M10ALFVP000J306   ALFA ROMEO          159    GO     non             

## 5. Vérifier et exporter

Les exports utilisent UTF-8 avec BOM et le séparateur `;`. Les valeurs absentes restent des champs vides. Pour les recharger : `pd.read_csv(chemin, sep=';', encoding='utf-8-sig', dtype={'cnit': 'string', 'tvv': 'string'})`.


In [6]:
assert not df.duplicated().any()
assert not df_analyse.isna().any().any()
assert not df[numeriques].lt(0).any().any()
assert df['masse_ordma_min'].le(df['masse_ordma_max']).all()
assert len(df_brut) == doublons + len(a_verifier) + len(df)
for col in df.select_dtypes(include=['object', 'string']).columns:
    assert df[col].dropna().eq(df[col].dropna().str.strip()).all()
    assert not df[col].dropna().eq('').any()

exports = {
    'vehicules_nettoyes.csv': df,
    'vehicules_analyse_conso_co2.csv': df_analyse,
    'lignes_a_verifier.csv': a_verifier,
    'bilan_qualite.csv': bilan.rename_axis('variable').reset_index(),
}
for nom, tableau in exports.items():
    chemin = sortie / nom
    tableau.to_csv(chemin, sep=';', encoding='utf-8-sig', index=False)
    relu = pd.read_csv(chemin, sep=';', encoding='utf-8-sig')
    assert relu.shape == tableau.shape
    print(f'{chemin} : {tableau.shape[0]} lignes, {tableau.shape[1]} colonnes')
print('Contrôles réussis.')


donnees_nettoyees/vehicules_nettoyes.csv : 20880 lignes, 26 colonnes
donnees_nettoyees/vehicules_analyse_conso_co2.csv : 20824 lignes, 12 colonnes
donnees_nettoyees/lignes_a_verifier.csv : 0 lignes, 27 colonnes
donnees_nettoyees/bilan_qualite.csv : 26 lignes, 7 colonnes
Contrôles réussis.
